In [1]:
import os
import hail as hl
import pandas as pd

In [2]:
def hail_init(chrom=None, log_prefix='get_vcf'):
    r'''Initialize Hail '''
    assert chrom in range(1,23), 'only autosomes allowed'
    n_slots = os.environ.get('NSLOTS', 1)
    chr_suffix = '' if chr is None else f'_chr{chrom}'
    WD = '/well/lindgren/UKBIOBANK/flassen/projects/KO/wes_ko_ukbb'
    hl.init(log=f'{WD}/logs/hail-{log_prefix}{chr_suffix}.log',
            default_reference='GRCh38',
            master=f'local[{n_slots}]')

hail_init(22)

Running on Apache Spark version 3.1.2
SparkUI available at http://compe041.hpc.in.bmrc.ox.ac.uk:4040
Welcome to
     __  __     <>__
    / /_/ /__  __/ /
   / __  / _ `/ / /
  /_/ /_/\_,_/_/_/   version 0.2.74-0c3a74d12093
LOGGING: writing to /well/lindgren/UKBIOBANK/flassen/projects/KO/wes_ko_ukbb/logs/hail-get_vcf_chr22.log


In [120]:
#!/usr/bin/env python3

# MISC functions
def recode_contigs():
    return {f"NC_00000{i}.{j}":f"chr{i}" for i in list(range(1, 23)) + ['X','Y'] for j in range(1,15)}

def get_table(input_path, input_type, min_partitions = None, cache=False):
    r'''Import mt/vcf/plink tables '''
    if input_type=='mt':
        mt = hl.read_matrix_table(input_path)
    elif input_type=='vcf':
        mt = hl.import_vcf(input_path, force_bgz=True, array_elements_required=False, min_partitions=min_partitions)
    elif input_type=='plink':
        mt = hl.import_plink(*[f'{input_path}.{x}' for x in ['bed','bim','fam']])
    if input_type!='mt' and cache:
        mt = mt.cache()
    mt = recalc_info(mt)
    return mt

def export_table(mt, out_prefix, out_type, checkpoint=False, format_fields_to_drop=[]):
    r'''Merge file with VEP info '''
    assert out_type in {'mt', 'vcf', 'plink'}
    if len(format_fields_to_drop) > 0:
        mt = mt.drop(*[f for f in format_fields_to_drop if f in mt.entry])
    if checkpoint:
        mt = mt.checkpoint(out_prefix+'.mt')
    if out_type=='mt':
        print(f'\nWriting to MT: {out_prefix}.mt\n')
        mt.write(out_prefix+'.mt')
    elif out_type=='vcf':
        out_vcf_path = out_prefix+'.vcf.bgz'
        print(f'\nExporting to VCF: {out_vcf_path}\n')
        if not hl.hadoop_is_file(out_vcf_path):
            metadata = get_vcf_metadata(format_fields_to_drop=format_fields_to_drop)
            hl.export_vcf(dataset=mt,
                          output=out_vcf_path,
                          parallel=None,
                          metadata = metadata)
        else:
            raise ValueError(f'Cannot export VCF because {out_vcf_path} already exists')
    elif out_type=='plink':
        print(f'\nExporting to PLINK: {out_prefix}.{{bed,bim,fam}}\n')
        mt = annotate_with_fam_fields(mt)
        if not any(hl.hadoop_is_file(out_prefix+s) for s in ['bed','bim','fam']): # don't overwrite if any files exist
            hl.export_plink(dataset=mt,
                            output=out_prefix,
                            fam_id=mt.fam_id,
                            pat_id=mt.pat_id,
                            mat_id=mt.mat_id,
                            is_female=mt.is_female,
                            varid=mt.rsid)
        else:
            raise ValueError(f'Cannot export to PLINK because at least one of {out_prefix}.{{bed,bim,fam}} already exists')


In [121]:
#!/usr/bin/env python3
#
#  QC functions
#
#' @description Compound heterozygos HAIL pipeline
#' @todo Integrate SAIGE-GENE+ / write to Wei
#' @DONE @todo Convert MatrixTable (sample, variant)-pairs to 'long' format
#' @DONE @todo Long format table should also contain variants found and their strand.
#' @DONE @todo Test burden_dosage function
#' @DONE @todo Create function that subsets variants of MODERATE impact
#' @DONE @todo hardy eq test for each variant
#' @todo add mt.repartition to different commands
#' @DONE @todo check worst consequence
#' @todo export ultra rare variants
#' @todo check which allele is included in VEP? What about MAF thresholds?


import hail as hl
import os
import argparse

#WD = '/well/lindgren/UKBIOBANK/flassen/projects/KO/wes_ko_ukbb'
#DATA_DIR = f'{WD}/data'



## QC PIPELINE
def filter_max_maf(mt, maf=None):
    r'''Filter to variants to have maf less than {maf}'''
    if maf is not None:
        mt = mt.filter_rows(hl.min(mt.info.AF)<=maf)
    return mt

def filter_min_maf(mt, maf=None):
    r'''Filter to variants to have maf gt {maf}'''
    #mt = mt.filter_rows(hl.min(mt.info.AF)>maf)
    if maf is not None:
        mt = mt.filter_rows(hl.min(mt.info.AF)>maf)
    return mt

def filter_max_mac(mt, mac=None):
    r'''Filter to variants to have maf less than {maf}'''
    if mac is not None:
        mt = mt.filter_rows(hl.min(mt.info.AC)<=mac)
    return mt

def filter_min_mac(mt, mac=None):
    r'''Filter to variants to have mac gt {mac}'''
    if mac is not None:
        mt = mt.filter_rows(hl.min(mt.info.AC)>mac)
    return mt

def filter_min_missing(mt, value = 0.05):
    r'''Filter variants to have less than {value} in genotype missigness"'''
    assert value >= 0 and value < 1
    missing = hl.agg.mean(hl.is_missing(mt.GT)) <= value
    mt = mt.filter_rows(missing)
    return mt

def translate_sample_ids(ht, from_app: int, to_app: int):
    r'''Translate sample IDs from one UKB application to another
    `from_app` and `to_app` are UKB application IDs. This function only supports
    translation in either direction between applications 11867 (Lindgren) and 12788 (McVean)
    '''
    from_app, to_app = int(from_app), int(to_app)
    valid_apps = {11867, 12788}
    assert (from_app in valid_apps) & (to_app in valid_apps), f'from_app and to_app must be in {valid_apps}'
    assert from_app!=to_app, 'from_app and to_app must be different'
    print(f'Mapping IDs from UKB app {from_app} to {to_app}')
    id_dict = hl.import_table('/well/lindgren/UKBIOBANK/nbaya/resources/ukb11867_to_ukb12788.sample_ids.txt',delimiter='\s+', key=f'eid_{from_app}')
    ht = ht.key_cols_by(s = id_dict[ht.s][f'eid_{to_app}'])
    undefined_ct = ht.aggregate_cols(hl.agg.sum(hl.is_missing(ht.s)))
    assert undefined_ct==0, f'[translate_sample_ids]: Not all sample IDs mapped perfectly ({undefined_ct}/{ht.count()} IDs are undefined)'
    return ht

def filter_to_european(mt):
    r'''Get white british (app 11867) /well/lindgren/UKBIOBANK/DATA/QC/ukb_sqc_v2.txt
    and genetically european from /well/lindgren/UKBIOBANK/laura/k_means_clustering_pcs/ukbb_genetically_european_k4_4PCs_self_rep_Nov2020.txt''' 
    ht = hl.import_table('/well/lindgren/UKBIOBANK/laura/k_means_clustering_pcs/ukbb_genetically_european_k4_4PCs_self_rep_Nov2020.txt',
        types={'eid': hl.tstr, 'genetically_european': hl.tint32}).key_by('eid')
    mt = mt.annotate_cols(eur = ht[mt.s].genetically_european)
    undefined_eur = mt.aggregate_cols(hl.agg.sum(hl.is_missing(mt.eur)))
    pre_filter_count = mt.count()
    if undefined_eur == pre_filter_count[1]:
        raise ValueError('[get_genetically_european]: IDs for genetically europeans does not match keys in MatrixTable!')
    if undefined_eur > 0:
        print(f'[get_genetically_european]: Not all samples IDs mapped perfectly ({undefined_eur}/{pre_filter_count[1]} IDs are undefined)')
    #mt = mt.filter_cols(mt.s[mt.eur])
    mt = mt.filter_cols(mt.eur == 1)
    post_filter_count = mt.count()
    print(f'[get_genetically_european]:{post_filter_count[1]}/{pre_filter_count[1]} IDs were included as genetically european.')
    return mt

def get_fam(app_id=12788, wes_200k_only=False):
    # use files created by the command:
    #   /well/lindgren/UKBIOBANK/nbaya/wes_200k/phase_ukb_wes/utils/make_fam.py --make_fam
    # github: https://github.com/nikbaya/phase_ukb_wes/blob/a5a97de8604d4ea1c7e7cfeecd9a1f57c2af2357/utils/make_fam.py
    assert app_id in {11867,12788}
    fam_path = f'/well/lindgren/UKBIOBANK/nbaya/resources/ukb{app_id}_{"wes_200k_" if wes_200k_only else ""}pedigree.fam'
    fam = hl.import_table(paths=fam_path, key='IID',types={f:'str' for f in ['FID','IID','PAT','MAT','SEX','PHEN']})
    return fam

def recalc_info(mt, maf=None, info_field='info', gt_field='GT'):
    r'''Recalculate INFO fields AF, AC, AN and keep sites with minor allele
    frequency > `maf The fields AF and AC are made into integers instead of 
    an array, only showing the AF and AC for the alt allele
    '''
    if info_field in mt.row:
        mt = mt.annotate_rows(
            info = mt[info_field].annotate(**hl.agg.call_stats(mt[gt_field],
                                                               mt.alleles)
                                               ).drop('homozygote_count')
            ) # recalculate AF, AC, AN, drop homozygote_count
    else:
        mt = mt.annotate_rows(**{info_field: hl.agg.call_stats(mt[gt_field], mt.alleles)})
    mt = mt.annotate_rows(
        **{info_field: mt[info_field].annotate(
            **{field:mt[info_field][field][1] for field in ['AF','AC']}
            )}
        ) # keep only the 2nd entries in AF and AC, which correspond to the alt alleles
    if maf is not None:
        mt = mt.filter_rows((mt[info_field].AF>maf) & (mt[info_field].AF<(1-maf)))
    return mt

def filter_to_unrelated(mt, get_related=False, maf=None):
    r'''Filter to samples in duos/trios, as defined by fam file'''
    fam = get_fam()
    fam = fam.filter(hl.is_defined(mt.cols()[fam.IID])) # need to filter to samples present in mt first before collecting FIDs
    fam_ct = fam.count()
    col_ct = mt.count_cols()
    print(f'\n{col_ct-fam_ct}/{col_ct} samples have IIDs missing from fam file') # samples with missing IIDs cannot be included in the related sample set
    iids = fam.IID.collect()
    offspring_fids = fam.filter(hl.literal(iids).contains(fam.PAT)
                                |hl.literal(iids).contains(fam.MAT)).FID.collect() # get the FIDs of offspring with at least one parent in the dataset
    fam = fam.filter(hl.literal(offspring_fids).contains(fam.FID)) # subset to samples which share FIDs with offspring
    if get_related:
        mt = mt.filter_cols(hl.is_defined(fam[mt.s])) # related
    else:
        mt = mt.filter_cols(~hl.is_defined(fam[mt.s])) # unrelated
    mt = recalc_info(mt=mt, maf=maf)
    return mt

def filter_hwe(mt, cut_off = 1e-12):
    mt =  mt.annotate_rows(hwe = hl.agg.hardy_weinberg_test(mt.GT))
    rows = mt.hwe.p_values >= cut_off
    mt = mt.filter_rows(rows)
    return mt

def get_vcf_metadata(info_fields_to_drop=[], format_fields_to_drop=['RNC','PL']):
    r'''Merge file with VEP info '''
    path = '/well/ukbb-wes/pvcf/oqfe/ukbb-wes-oqfe-pvcf-chr1.vcf.gz' # use a VCF that has the same fields and is unlikely to be deleted
    metadata = hl.get_vcf_metadata(path)
    for f in info_fields_to_drop:
        metadata['info'].pop(f)
    for f in format_fields_to_drop:
        metadata['format'].pop(f)
    return metadata

def annotate_with_fam_fields(mt):
    r'''Annotate `mt` with FID (fam_id), PAT (pat_id), MAT (mat_id), SEX (is_female)from fam file with all UKB samples'''
    fam = get_fam(wes_200k_only=False)
    mt = mt.annotate_cols(fam_id = fam[mt.s].FID,
                          pat_id = fam[mt.s].PAT,
                          mat_id = fam[mt.s].MAT,
                          is_female = hl.if_else(fam[mt.s].SEX=='2',
                                                 True,
                                                 hl.if_else(fam[mt.s].SEX=='1',
                                                            False,
                                                            hl.null(hl.tbool))))
    return mt



def remake_rsid(mt, delim = '_'):
    r'''Re-create rsids based on current locus and alleles'''
    ids = hl.delimit([mt.locus.contig, hl.str(mt.locus.position), mt.alleles[0], mt.alleles[1]], delim)
    mt = mt.annotate_rows(rsid = ids)
    return mt


            


In [161]:
def annotate_vep(mt, vep_path):
    r'''Annotate matrix table with VEP consequence from external file.'''
    #print(f'Annotating with VEP file: {vep_path}')
    
    # Open file containing VEP fields
    with open('data/vep/vep_fields.txt', 'r') as file:
        fields = file.read().strip().split(',')
    ht = hl.import_vcf(vep_path).rename({'info':'vep'}) 
    
    # Add VEP fields by iteration
    for i in range(len(fields)):
        ht = ht.annotate_rows(
            vep=ht.vep.annotate(
                col=ht.vep.CSQ.map(lambda x: (x.split('\\|')[i]))[0]
                ).rename({'col':f'{fields[i]}'})
        )
    
    # Most severe variant consequence
    ht = ht.annotate_rows(vep = ht.vep.annotate(most_severe_consequence = ht.vep.Consequence.split('&')[0]))
    
    # Extract various categories annotations and change type
    ht = ht.annotate_rows(vep = ht.vep.annotate(sift_pred = ht.vep.SIFT_pred.split('&')[0]))
    ht = ht.annotate_rows(vep = ht.vep.annotate(polyphen2_hdiv_pred = ht.vep.Polyphen2_HDIV_pred.split('&')[0]))
    ht = ht.annotate_rows(vep = ht.vep.annotate(polyphen2_hvar_pred = ht.vep.Polyphen2_HVAR_pred.split('&')[0]))
    ht = ht.annotate_rows(vep = ht.vep.annotate(cadd_phred_score = hl.parse_float(ht.vep.CADD_phred)))
    ht = ht.annotate_rows(vep = ht.vep.annotate(revel_score = hl.parse_float(ht.vep.REVEL_score)))
    
    # Define protein truncating variants
    ptv = hl.set(["transcript_ablation", "splice_acceptor_variant",
              "splice_donor_variant", "stop_gained", "frameshift_variant"])
    
    # Define missense variation
    missense = hl.set(["stop_lost", "start_lost", "transcript_amplification",
                   "inframe_insertion", "inframe_deletion", "missense_variant",
                   "protein_altering_variant", "splice_region_variant"])
    
    # Define synonymous
    synonymous = hl.set(["incomplete_terminal_codon_variant", "stop_retained_variant", "synonymous_variant"])
    
    # Define non coding variation
    non_coding = hl.set(["coding_sequence_variant", "mature_miRNA_variant", "5_prime_UTR_variant",
              "3_prime_UTR_variant", "non_coding_transcript_exon_variant", "intron_variant",
              "NMD_transcript_variant", "non_coding_transcript_variant", "upstream_gene_variant",
              "downstream_gene_variant", "TFBS_ablation", "TFBS_amplification", "TF_binding_site_variant",
              "regulatory_region_ablation", "regulatory_region_amplification", "feature_elongation",
              "regulatory_region_variant", "feature_truncation", "intergenic_variant"])
    
    # Create categories for downstream analysis
    ht = ht.annotate_rows(vep = ht.vep.annotate(consequence_category = 
        hl.case().when(ptv.contains(ht.vep.most_severe_consequence), "ptv")
             .when(missense.contains(ht.vep.most_severe_consequence) & 
                   (~hl.is_defined(ht.vep.cadd_phred_score) | 
                    ~hl.is_defined(ht.vep.revel_score)), "other_missense")                                   
             .when(missense.contains(ht.vep.most_severe_consequence) & 
                   (ht.vep.cadd_phred_score >= 20) & 
                   (ht.vep.revel_score >= 0.6), "damaging_missense") 
             .when(missense.contains(ht.vep.most_severe_consequence), "other_missense")
             .when(synonymous.contains(ht.vep.most_severe_consequence), "synonymous")
             .when(non_coding.contains(ht.vep.most_severe_consequence), "non_coding")
             .default("NA")
    ))
                                                
    # combine with matrix table
    mt = mt.annotate_rows(vep = ht.index_rows(mt.locus, mt.alleles).vep.drop('CSQ'))
    return(mt)

def filter_vep(mt, field, conditions):
    r'''Filter VEP field by condition(s) '''
    assert isinstance(conditions, list)
    assert field in list(mt.vep)
    mt = mt.filter_rows(hl.literal(set(conditions)).contains(mt.vep[field]))
    return mt


def annotate_phased_entries(mt):
    r'''Annotates alleles that have the alternate allele on either first or second strand.'''
    mt = mt.annotate_entries(a0_alt = mt.GT ==  hl.parse_call('1|0'))
    mt = mt.annotate_entries(a1_alt = mt.GT ==  hl.parse_call('0|1'))
    mt = mt.annotate_entries(a_homo = mt.GT ==  hl.parse_call('1|1'))
    return mt


def construct_phased_dosage_mt(mt, gene_field = 'Gene'):
    r''' Returns matrix table that contains dosage information from phased geneotypes.
    0: two refererence alleles in locus,
    1: one alternate allele on either strand in a locus, 
    2: two alternate allele on either strand in a locus (either as homozygous or compound heterozygous)
    '''
    mt = annotate_phased_entries(mt)
    knockout =  (mt.a0_alt & mt.a1_alt) | mt.a_homo # ((mt.a0_alt & mt.a1_alt) | mt.a_homo ) & (mt.vep.consequence_category == hl.literal("ptv"))
    heterozygous = (mt.a0_alt | mt.a1_alt)
    burden_mt = (
        mt 
        .group_rows_by(mt.vep[gene_field])
        .aggregate(dosage = hl.if_else( hl.agg.any(knockout), 2, 
                            hl.if_else( hl.agg.any(heterozygous), 1, 0 )))
        #.burden_mt.filter_entries(burden_mt.entries != 0)
    )
    return burden_mt

def construct_summary_mt(mt, gene_field = 'Gene'):
    r''' Returns matrix table that contains dosage information from phased geneotypes.
    0: two refererence alleles in locus,
    1: one alternate allele on either strand in a locus, 
    2: two alternate allele on either strand in a locus (either as homozygous or compound heterozygous)
    '''
    
    # setup booleans
    mt = annotate_phased_entries(mt)
    homozygous = mt.a_homo
    compound_heterozygous = mt.a0_alt & mt.a1_alt
    heterozygous = mt.a0_alt | mt.a1_alt

    # aggregate onto gene level
    ht = (
        mt 
        .group_rows_by(mt.vep[gene_field])
        .aggregate(ko = hl.if_else( hl.agg.any(compound_heterozygous & ~homozygous) , 'CH', 
                        hl.if_else( hl.agg.any(homozygous & ~compound_heterozygous) , 'HO',
                        hl.if_else( hl.agg.any(compound_heterozygous & homozygous) , 'CH+HO',
                        hl.if_else( hl.agg.any(heterozygous) , 'HE', '')))))
        .filter_entries(mt.entries != '')
    )
    
    return ht

def extract_gene_ko_rows(burden_mt):
    assert ko in list(burden_mt)
    entries = burden_mt.entries()
    entries = entries.filter(~hl.literal('').contains(entries.ko))
    return entries

def extract_knockout_samples(mt, gene_field ='Gene', keep = ['HO','CH+HO']):
    r''' Collapses variants into genes for each individual, and returns
    a with three columns containing:
    1: The gene that was used for collapsing
    2: The current individual 
    4: Either CH (Compound heterozygous), HO (Homozygous) or CH+HO or H (Heterozygous).
       To avoid too long matrices, this is specified by the keep column.
    3: The variant and corresponding genotype for the variant in the sample,
      e.g. "chr22_50604850_G_A=1|1,chr22_50606762_C_T=1|1".
    '''
    
    # Parameters for KO/CH
    mt = annotate_phased_entries(mt)
    homozygous = mt.a_homo
    compound_heterozygous = mt.a0_alt & mt.a1_alt
    heterozygous = mt.a0_alt | mt.a1_alt

    # aggregate the gene-level
    ht1 = (mt 
          .group_rows_by(mt.vep[gene_field])
          .aggregate(ko = hl.if_else( hl.agg.any(compound_heterozygous & ~homozygous) , 'CH', 
                          hl.if_else( hl.agg.any(homozygous & ~compound_heterozygous) , 'HO',
                          hl.if_else( hl.agg.any(compound_heterozygous & homozygous) , 'CH+HO',
                          hl.if_else( hl.agg.any(heterozygous) , 'HE', '')))))
          .filter_entries(mt.entries != ''))

    # generate rsid to gt entries
    mt = mt.annotate_entries(rsid_entry = mt.rsid)
    mt = mt.annotate_entries(rsid_gt = hl.delimit([mt.rsid_entry, hl.str(mt.GT)], '='))

    # annotate only ko entries
    ht2 = (
            mt 
            .group_rows_by(mt.vep[gene_field])
            .aggregate(rsid = hl.agg.filter(mt.a_homo | (mt.a0_alt & mt.a1_alt), hl.agg.collect(mt.rsid_gt)))
    )

    # combine entries
    ent1 = ht1.entries()
    ent2 = ht2.entries()
    combined = ent1.annotate(genotypes = ent2[ent1[gene_field], ent1.s])
    combined = combined.filter(hl.set(keep).contains(combined.ko))
    line_merge = hl.delimit(combined.genotypes.rsid, ',')
    combined = combined.annotate(genotypes = combined.genotypes.annotate(rsid = line_merge))
    return combined


def count_alleles(mt):
    r'''Count up alleles in a vector (singleton AC, not singletons AC, total AC)'''
    mt = mt.filter_rows(mt.info.AC > 0)
    d = mt.aggregate_rows(hl.agg.counter(mt.info.AC))
    if 1 in d.keys():
        n_singletons_ac = d[1]
    else:
        n_singletons_ac = 0
    n_not_singletons_ac = sum([value*key for key, value in d.items() if key != 1])
    return [n_singletons_ac, n_not_singletons_ac, n_singletons_ac + n_not_singletons_ac]

def count_genes(mt):
    r'''Collapse variants into genes and count affected genes'''
    d = mt.aggregate_entries(hl.agg.group_by(mt.vep.Gene, hl.agg.count_where(mt.GT.is_non_ref())))
    n_genes = len([(key, value) for key, value in d.items() if value != 0])
    return n_genes

def summarize_variants(mt, what = 'ptv', vep_field = 'consequence_category'):
    r'''Count up singletons, non singletons, total and genes affected by variants'''
    ht = mt.filter_rows(hl.literal(set([what])).contains(mt.vep[vep_field]))
    ht_alleles = count_alleles(ht)
    ht_genes = count_genes(ht)
    out = ht_alleles + [ht_genes]
    return out

def gene_burden_annotations_per_sample(mt, gene_field = 'Gene'):
    mt = mt.group_rows_by(
        Gene = mt.vep.Gene#,
        #consequence_category = mt.vep.consequence_category
        ).aggregate(n = hl.agg.count_where(mt.GT.is_non_ref()))
    return mt

def gene_burden_category_annotations_per_sample(mt, gene_field = 'Gene'):
    mt = mt.group_rows_by(
        Gene = mt.vep.Gene,
        consequence_category = mt.vep.consequence_category
        ).aggregate(n = hl.agg.count_where(mt.GT.is_non_ref()))
    return mt

def calc_p_ko(mt):
    '''
    Annotates entries with P(Knockout). Requires, that fields are
    already annotated with "singletons" count, and "dosage". 
    '''
    ko_mt = mt.annotate_entries(
        pKO = hl.if_else(
            mt.dosage == 2, 1, # knockout
            hl.if_else(
                mt.dosage == 1, 
                hl.if_else(mt.singletons >= 1, 1 - (1/2)**mt.singletons, 0), # one phased hetz
                hl.if_else(mt.singletons >= 2, 1 - 2*(1/2)**mt.singletons, 0), # zero phased hetz
            )
        )
    )
    return ko_mt

def get_prob_ko_matrix(mt_phased, mt_unphased, fields_drop = ['dosage','sigletons']):
    
    '''
    Annotates entries with P(Knockout). Requires a phased matrix and an unphased
    matrix that only contains singletons.
    '''
    
    # setup variables
    mt1 = mt_phased # contains phased non-singletons
    mt2 = mt_unphased # contains unphased singletons
    
    # add check to ensure that mt1 is phased
    # add check to ensure that mt2 only contains singletons
    
    # Determine probability of being KO given singletons and phased hetz
    mt1_burden = construct_phased_dosage_mt(mt1)
    mt2_burden = gene_burden_annotations_per_sample(mt2)
    mt_ko = mt1_burden.annotate_entries(singletons = mt2_burden[(mt1_burden.Gene, mt1_burden.s)].n)
    mt_ko = mt_ko.annotate_entries(singletons = hl.if_else(~hl.is_missing(mt_ko.singletons), mt_ko.singletons, 0 ))
    mt_ko = calc_p_ko(mt_ko)

    # drop not needed rows
    mt_ko = mt_ko.drop(*[f for f in fields_drop if f in mt_ko.entry])
    #mt_ko_entries = mt_ko.entries()
    #mt_ko_entries = mt_ko_entries.filter(~hl.is_missing(mt_ko_entries.pKO))
    return mt_ko



def get_dummy_by_dp(mt1, mt2, chrom):

    # get probability matrix
    pmt = get_prob_ko_matrix(mt1, mt2, ["dosage","singletons"])
    pmt = pmt.annotate_entries(DP = pmt.pKO*2) # multiply probability by 2 (dosage encoded [0:2])
    pmt = pmt.drop('pKO')

    # create fake loci
    pmt = pmt.annotate_rows(locus = hl.parse_locus('chr' + str(chrom) + ':1'))
    pmt = pmt.annotate_rows(alleles = hl.literal(['X','Y']))
    pmt = pmt.annotate_rows(rsid = pmt.Gene)
    pmt = pmt.key_rows_by(pmt.locus, pmt.alleles)
    pmt = pmt.drop('Gene')
    return pmt
    




In [123]:
# get non singletons (note: individuals are on 12788)
mt1 = get_table('data/tmp/ukb_wes_200k_phased_tmp_chr22.1of1.vcf.gz', 'vcf')
mt1 = translate_sample_ids(mt1, from_app='12788', to_app='11867') # translate to lindgren app ID
mt1 = annotate_vep(mt1, vep_path = 'data/vep/output/ukb_wes_200k_vep_chr22.vcf')
mt1 = filter_min_missing(mt1, 0.05)
mt1 = filter_vep(mt1, 'consequence_category',['damaging_missense','ptv'])



Mapping IDs from UKB app 12788 to 11867


2021-09-03 17:04:49 Hail: INFO: Reading table without type imputation
  Loading field 'eid_11867' as type str (not specified)
  Loading field 'eid_12788' as type str (not specified)
2021-09-03 17:04:49 Hail: INFO: Coerced almost-sorted dataset


In [124]:
# get singletons (Note individuals are already 11867)
mt2 = get_table('data/unphased/tmp/ukb_wes_200k_filtered_2samples_chr22.vcf.bgz', 'vcf')
mt2 = filter_max_mac(mt2, 1) # get singletons
mt2 = filter_min_missing(mt2, 0.05)
mt2 = annotate_vep(mt2, vep_path = 'data/vep/full/ukb_wes_200k_full_vep_chr22.vcf')
mt2 = mt2.filter_rows(hl.agg.any(mt2.GT.is_non_ref()))
mt2 = filter_vep(mt2, 'consequence_category',['damaging_missense','ptv'])

#mtB.show()
#mtB.show()
#mtB.filter_rows(hl.agg.sum(mtB.n) > 0).show()

In [125]:
#mt2 = get_table('data/unphased/tmp/ukb_wes_200k_filtered_2samples_chr22.vcf.bgz', 'vcf')
#mt2.describe()

In [126]:
#hl.parse_locus('chr' + str(chrom) + ':1')
#

SyntaxError: invalid syntax (<ipython-input-126-eec211f2418d>, line 2)

In [162]:
out = get_dummy_by_dp(mt1, mt2, 22)
out.show()
#export_table(out, out_prefix = 'dummy_test', out_type = 'vcf')

2021-09-03 17:43:38 Hail: INFO: Coerced almost-sorted dataset
2021-09-03 17:43:47 Hail: INFO: Coerced sorted dataset
2021-09-03 17:43:48 Hail: INFO: Coerced almost-sorted dataset
2021-09-03 17:43:50 Hail: INFO: Coerced almost-sorted dataset
2021-09-03 17:43:57 Hail: INFO: Ordering unsorted dataset with network shuffle
2021-09-03 17:43:58 Hail: INFO: Coerced sorted dataset
2021-09-03 17:43:59 Hail: INFO: Coerced sorted dataset
2021-09-03 17:44:11 Hail: INFO: Ordering unsorted dataset with network shuffle
2021-09-03 17:44:12 Hail: INFO: Coerced sorted dataset


,,,
,,'1705888','3544034'
locus,alleles,DP,DP
locus<GRCh38>,array<str>,float64,float64
chr22:1,"[""X"",""Y""]",0.00e+00,0.00e+00
chr22:1,"[""X"",""Y""]",0.00e+00,0.00e+00
chr22:1,"[""X"",""Y""]",0.00e+00,0.00e+00
chr22:1,"[""X"",""Y""]",0.00e+00,0.00e+00
chr22:1,"[""X"",""Y""]",0.00e+00,0.00e+00
chr22:1,"[""X"",""Y""]",0.00e+00,0.00e+00
chr22:1,"[""X"",""Y""]",0.00e+00,0.00e+00


In [149]:
mt2.alleles[0].show()

2021-09-03 17:12:43 Hail: INFO: Coerced sorted dataset
2021-09-03 17:12:44 Hail: INFO: Coerced sorted dataset


,,
locus,alleles,
locus<GRCh38>,array<str>,str
chr22:25203882,"[""G"",""A""]","""G"""
chr22:25359975,"[""C"",""G""]","""C"""
chr22:26027319,"[""C"",""T""]","""C"""
chr22:32239053,"[""G"",""T""]","""G"""
chr22:42130692,"[""G"",""A""]","""G"""


In [112]:
mt1

mt1

In [35]:
# Determine probability of being KO given singletons and phased hetz
mt1_burden = construct_phased_dosage_mt(mt1)
mt2_burden = gene_burden_annotations_per_sample(mt2)
mt_ko = mt1_burden.annotate_entries(singletons = mt2_burden[(mt1_burden.Gene, mt1_burden.s)].n)
mt_ko = calc_ko_prob(mt_ko)
#mt_ko.filter_rows(hl.agg.any(mt_ko.singletons > 0)).show()
mt_ko = mt_ko.drop('dosage')
mt_ko = mt_ko.drop('singletons')
mt_ko_entries = mt_ko.entries()
mt_ko_entries = mt_ko_entries.filter(~hl.is_missing(mt_ko_entries.pKO))


2021-09-02 18:17:14 Hail: INFO: Coerced sorted dataset
2021-09-02 18:17:15 Hail: INFO: Coerced almost-sorted dataset
2021-09-02 18:17:16 Hail: INFO: Coerced almost-sorted dataset
2021-09-02 18:17:19 Hail: INFO: Ordering unsorted dataset with network shuffle
2021-09-02 18:17:20 Hail: INFO: Coerced sorted dataset
2021-09-02 18:17:21 Hail: INFO: Coerced sorted dataset
2021-09-02 18:17:25 Hail: INFO: Ordering unsorted dataset with network shuffle


,,
Gene,s,pKO
str,str,float64
"""ENSG00000008735""","""1705888""",5.00e-01
"""ENSG00000008735""","""3544034""",1.00e+00
"""ENSG00000015475""","""1705888""",5.00e-01
"""ENSG00000015475""","""3544034""",5.00e-01
"""ENSG00000054611""","""1705888""",1.00e+00
"""ENSG00000054611""","""3544034""",1.00e+00
"""ENSG00000056487""","""1705888""",1.00e+00
"""ENSG00000056487""","""3544034""",1.00e+00


In [30]:


z = x.annotate_entries(k = y[(x.Gene, x.s)].n)
z.annotate_entries(
    pKO = hl.if_else(
        z.dosage == 2, 1, # knockout
        hl.if_else(
            z.dosage == 1, 
            hl.if_else(z.k >= 1, 1 - (1/2)**z.k, 0), # one phased hetz
            hl.if_else(z.k >= 2, 1 - 2*(1/2)**z.k, 0), # zero phased hetz
        )
    )

).show()

In [ ]:
#res = hl.vep(mtB, config='utils/configs/vep_env_pick.json')

In [ ]:
#res.vep.variant_class.show()

In [6]:
def write_saige_step1(mt):
    # write matrix table variants x samples (that also contains singletons)
    
    return None

def write_saige_step2(mt):
    # using variant x samples, collapse to knockouts (1 or 0)
    # annotate with singletons
    # probabilistic model to output entries gene - individual knockout probability
    return None

In [9]:
# Setup data and do annotation
#mt = get_table('data/mt/test_write.mt', 'mt')
#mt = hl.import_vcf('data/phased/ukb_wes_200k_phased_chr22.1of1.vcf.gz', force_bgz=True, array_elements_required=False)
#export_table(mt, out_prefix = 'data/mt/ukb_wes_200k_phased_chr22.1of1', out_type = 'mt')
#mt = annotate_vep(mt, vep_path = 'data/vep/output/ukb_wes_200k_vep_chr22.vcf')
#mt = remake_rsid(mt)
#mt = recalc_info(mt)
#print('info recalced')
#mt.info.AC.show()
#mt.filter_rows(mt.info.AC == 1).info.show()


In [6]:
# Setup data and do annotation
mt = get_table('data/tmp/ukb_wes_200k_phased_tmp_chr22.1of1.vcf.gz', 'vcf')
mt = annotate_vep(mt, vep_path = 'data/vep/output/ukb_wes_200k_vep_chr22.vcf')
# what about missigness filter?
#mt.aggregate_rows(hl.agg.counter(mt.vep.consequence_category))
#export_table(mt, out_prefix = 'data/tmp/ukb_wes_200k_phased_tmp_chr22.1of1', out_type = 'mt')
#export_table(mt, )
#mt = remake_rsid(mt)

# summarize variants
#summary_ptv = summarize_variants(mt, 'ptv')
#summary_missense = summarize_variants(mt, 'damaging_missense')

# QC steps
#mt_eur = filter_to_european(mt)
#mt_eur = filter_hwe(mt_eur)

#summary_eur_ptv = summarize_variants(mt, 'ptv')
#summary_eur_missense = summarize_variants(mt, 'damaging_missense')

# Construct knockout matrix
#burden_mt = construct_phased_dosage_mt(mt)


# check all samples, later we subset to europeans

Annotating with VEP file: data/vep/output/ukb_wes_200k_vep_chr22.vcf


In [41]:
recode = {f"NC_00000{i}.{j}":f"chr{i}" for i in list(range(1, 23)) + ['X','Y'] for j in range(1,15)}

In [42]:
db = hl.import_vcf('/well/lindgren/flassen/ressources/dbsnp/GRCh38/GCF_000001405.39.gz', force_bgz=True, reference_genome='GRCh38', contig_recoding=recode)

2021-08-25 15:51:11 Hail: WARN: expected input file 'file:/well/lindgren/flassen/ressources/dbsnp/GRCh38/GCF_000001405.39.gz' to end in .vcf[.bgz, .gz]


In [ ]:
db.show()

In [54]:
ht.show()

,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,
genotyping.array,Batch,Plate.Name,Well,Cluster.CR,dQC,Internal.Pico..ng.uL.,Submitted.Gender,Inferred.Gender,X.intensity,Y.intensity,Submitted.Plate.Name,Submitted.Well,sample.qc.missing.rate,heterozygosity,heterozygosity.pc.corrected,het.missing.outliers,putative.sex.chromosome.aneuploidy,in.kinship.table,excluded.from.kinship.inference,excess.relatives,in.white.British.ancestry.subset,used.in.pca.calculation,PC1,PC2,PC3,PC4,PC5,PC6,PC7,PC8,PC9,PC10,PC11,PC12,PC13,PC14,PC15,PC16,PC17,PC18,PC19,PC20,PC21,PC22,PC23,PC24,PC25,PC26,PC27,PC28,PC29,PC30,PC31,PC32,PC33,PC34,PC35,PC36,PC37,PC38,PC39,PC40,in.Phasing.Input.chr1_22,in.Phasing.Input.chrX,in.Phasing.Input.chrXY
str,str,str,str,float64,float64,float64,str,str,float64,float64,str,str,float64,float64,float64,int32,int32,int32,int32,int32,int32,int32,float64,float64,float64,float64,float64,float64,float64,float64,float64,float64,float64,float64,float64,float64,float64,float64,float64,float64,float64,float64,float64,float64,float64,float64,float64,float64,float64,float64,float64,float64,float64,float64,float64,float64,float64,float64,float64,float64,float64,float64,int32,int32,int32
"""UKBB""","""Batch_b001""","""SMP4_0012258""","""A01""",9.92e+01,9.60e-01,3.31e+01,"""M""","""M""",5.08e+02,5.78e+02,"""SMP4_0012258""","""A01""",5.75e-03,1.88e-01,1.90e-01,0,0,0,0,0,1,1,-1.22e+01,5.39e+00,-1.28e+00,8.42e-01,-5.27e+00,-1.79e+00,3.11e+00,-2.63e+00,2.39e+00,3.08e-01,3.33e+00,-1.97e+00,-6.05e-01,-2.04e+00,5.49e-01,-3.53e+00,8.95e-01,-5.94e+00,1.00e+00,4.44e+00,-1.85e+00,-1.67e+00,-6.55e-01,-2.18e+00,8.97e-01,-2.75e-01,-1.37e+00,-1.27e+00,-8.73e-01,2.37e+00,1.12e+00,6.14e-02,-4.38e+00,5.27e-01,-2.48e+00,2.27e+00,-1.18e+00,-4.46e+00,2.25e+00,-3.55e+00,1,1,1
"""UKBB""","""Batch_b001""","""SMP4_0012258""","""A02""",9.96e+01,9.70e-01,3.03e+01,"""F""","""F""",6.44e+02,2.81e+02,"""SMP4_0012258""","""A02""",1.66e-03,1.89e-01,1.91e-01,0,0,0,0,0,1,1,-1.30e+01,6.42e+00,-1.83e-01,2.93e+00,-5.89e+00,9.41e-01,1.14e+00,-1.98e+00,-2.70e+00,2.51e+00,2.36e+00,-1.22e+00,-1.55e+00,5.16e-01,5.99e-01,8.27e-02,7.98e-01,3.01e-01,4.67e+00,-3.97e+00,3.13e+00,-2.29e+00,-8.66e-01,4.52e+00,2.05e+00,-2.69e+00,3.39e-01,-8.46e-01,-2.69e+00,3.68e+00,6.77e-01,-3.43e+00,1.80e+00,-2.22e+00,-5.30e-01,-1.54e+00,1.26e+00,4.30e-01,2.31e+00,-1.56e+00,1,1,1
"""UKBB""","""Batch_b001""","""SMP4_0012258""","""A03""",9.97e+01,9.80e-01,3.90e+01,"""F""","""F""",7.22e+02,3.12e+02,"""SMP4_0012258""","""A03""",8.75e-04,1.89e-01,1.90e-01,0,0,0,0,0,1,1,-1.15e+01,3.48e+00,-1.15e+00,3.08e+00,7.65e+00,-9.13e-01,-1.55e+00,1.48e+00,-1.21e+00,1.06e+00,5.13e-01,1.01e+00,-2.29e+00,1.62e+00,-5.36e-02,-4.49e+00,1.09e+00,2.59e-02,-2.26e-01,-7.79e-01,-2.26e+00,-3.68e+00,2.57e+00,-4.02e+00,-1.24e+00,-2.52e+00,3.50e-01,1.62e+00,-1.21e+00,-1.20e+00,5.44e+00,1.76e+00,-1.21e+00,-6.51e+00,5.32e+00,-3.93e+00,1.38e+00,-3.92e+00,-1.93e+00,4.43e+00,1,1,1
"""UKBB""","""Batch_b001""","""SMP4_0012258""","""A04""",9.97e+01,9.80e-01,3.91e+01,"""F""","""F""",7.70e+02,3.30e+02,"""SMP4_0012258""","""A04""",6.87e-04,1.90e-01,1.91e-01,0,0,0,0,0,1,1,-1.21e+01,4.03e+00,-9.88e-01,7.50e-01,-2.36e+00,4.32e-01,-5.34e-01,-6.54e-01,-6.60e+00,-1.53e+00,-9.49e-01,2.92e-01,2.91e+00,1.79e+00,-8.68e-01,-1.42e+00,-2.36e+00,1.10e+00,9.54e-01,9.88e-02,-3.53e+00,-1.17e+00,-1.34e-01,-3.00e+00,1.79e-01,1.63e+00,5.92e+00,-1.87e+00,3.99e-01,-9.17e-01,2.48e+00,-2.46e+00,-6.04e+00,1.25e-01,-5.70e-01,1.38e+00,-3.87e+00,-1.76e+00,1.57e-01,-1.10e+00,1,1,1
"""UKBB""","""Batch_b001""","""SMP4_0012258""","""A05""",9.96e+01,9.80e-01,4.83e+01,"""F""","""F""",7.48e+02,3.35e+02,"""SMP4_0012258""","""A05""",1.87e-03,1.89e-01,1.91e-01,0,0,0,0,0,1,1,-1.22e+01,3.51e+00,-1.63e+00,-1.23e+00,-5.35e+00,3.82e+00,5.79e-01,-1.17e+00,1.02e+00,-7.95e-01,-1.58e-02,-1.62e+00,-1.20e+00,-2.02e+00,-6.78e-02,6.50e-01,3.05e+00,2.61e+00,-3.26e-01,-4.37e+00,-8.88e-01,-5.49e+00,-1.20e+00,1.79e+00,4.82e+00,3.70e-01,-7.44e+00,-1.07e-02,-2.32e+00,-2.18e+00,3.5

In [19]:
# stats needed
# * How many rare pLOF variants?
# * How many singletons
# * How many genes affected?
# * How many PTVs?
# * How many damaging missense?

summary_ptv = summarize_variants(mt, 'ptv')
print(str(summary_ptv[0]) + ' singletons PTVs')
print(str(summary_ptv[1]) + ' non-singletons PTVs')
print(str(summary_ptv[2]) + ' total PTVs')
print(str(summary_ptv[2]) + ' genes (PTVs)')

summary_missense = summarize_variants(mt, 'damaging_missense')
print(str(summary_missense[0]) + ' singletons damaging_missense')
print(str(summary_missense[1]) + ' non-singletons damaging_missense')
print(str(summary_missense[2]) + ' total damaging_missense')
print(str(summary_missense[2]) + ' genes (damaging_missense)')

2021-08-15 13:56:45 Hail: INFO: Coerced almost-sorted dataset
2021-08-15 13:56:45 Hail: INFO: Coerced almost-sorted dataset
2021-08-15 13:56:52 Hail: INFO: Coerced almost-sorted dataset
2021-08-15 13:56:53 Hail: INFO: Coerced almost-sorted dataset


0 singletons PTVs
6 non-singletons PTVs
6 total PTVs
6 genes (PTVs)


2021-08-15 13:57:01 Hail: INFO: Coerced almost-sorted dataset
2021-08-15 13:57:01 Hail: INFO: Coerced almost-sorted dataset
2021-08-15 13:57:07 Hail: INFO: Coerced almost-sorted dataset
2021-08-15 13:57:08 Hail: INFO: Coerced almost-sorted dataset


5 singletons damaging_missense
2 non-singletons damaging_missense
7 total damaging_missense
7 genes (damaging_missense)


,Singletons,non-singletons,total AC,Genes
ptv,0,6,6,2
missense,5,2,7,5


In [17]:
ko_mt = extract_knockout_samples(mt)
ko_mt.show()

2021-08-15 13:44:05 Hail: INFO: Coerced almost-sorted dataset
2021-08-15 13:44:05 Hail: INFO: Coerced almost-sorted dataset
2021-08-15 13:44:08 Hail: INFO: Ordering unsorted dataset with network shuffle
2021-08-15 13:44:08 Hail: INFO: Coerced almost-sorted dataset
2021-08-15 13:44:09 Hail: INFO: Coerced almost-sorted dataset
2021-08-15 13:44:12 Hail: INFO: Ordering unsorted dataset with network shuffle


,,,
,,,genotypes
Gene,s,ko,rsid
str,str,str,str
"""ENSG00000008735""","""2702437""","""HO""","""chr22_50604850_G_A=1|1,chr22_50606762_C_T=1|1,chr22_50607005_C_T=1|1,chr22_50610353_T_G=1|1"""
"""ENSG00000054611""","""2702437""","""HO""","""chr22_46878742_C_T=1|1"""
"""ENSG00000054611""","""5710601""","""HO""","""chr22_46878742_C_T=1|1"""
"""ENSG00000056487""","""2702437""","""HO""","""chr22_44913773_C_T=1|1"""
"""ENSG00000056487""","""5710601""","""HO""","""chr22_44913773_C_T=1|1"""
"""ENSG00000069998""","""2702437""","""HO""","""chr22_17149596_C_A=1|1"""
"""ENSG00000069998""","""5710601""","""HO""","""chr22_17159155_G_A=1|1"""


In [14]:
# Stats needed
# * How many knockouts?
# * How many CH Knockouts? 
# * How many HO knockouts?
ko_mt = extract_knockout_samples(mt)


2021-08-15 13:42:48 Hail: INFO: Coerced almost-sorted dataset
2021-08-15 13:42:49 Hail: INFO: Coerced almost-sorted dataset
2021-08-15 13:42:50 Hail: INFO: Coerced almost-sorted dataset
2021-08-15 13:42:53 Hail: INFO: Ordering unsorted dataset with network shuffle


,,
,'5710601','2702437'
Gene,ko,ko
str,str,str
"""""","""""",""""""
"""ENSG00000008735""","""HE""","""HO"""
"""ENSG00000015475""","""HE""","""HE"""
"""ENSG00000025770""","""""",""""""
"""ENSG00000040608""","""""",""""""
"""ENSG00000054611""","""HO""","""HO"""
"""ENSG00000056487""","""HO""","""HO"""


In [ ]:
# a function that summarizes variants
mt.aggregate_rows(hl.agg.counter(mt.vep.consequence_category))


In [7]:
mt.show()

2021-08-13 14:36:14 Hail: INFO: Coerced almost-sorted dataset
2021-08-13 14:36:16 Hail: INFO: Coerced almost-sorted dataset


,,,
,,'5710601','2702437'
locus,alleles,GT,GT
locus<GRCh38>,array<str>,call,call
chr22:15528182,"[""C"",""T""]",0|0,0|0
chr22:15528188,"[""C"",""T""]",0|0,0|0
chr22:15528194,"[""G"",""A""]",0|0,0|0
chr22:15528206,"[""G"",""A""]",0|0,0|0
chr22:15528267,"[""A"",""G""]",0|0,0|0
chr22:15528300,"[""A"",""G""]",0|0,0|0
chr22:15528306,"[""T"",""A""]",0|0,0|0


In [118]:
# Note, still lots of NA's since most severe consequence has not been selected.
# mt.aggregate_rows(hl.agg.counter(mt.vep.Consequence))
mt.aggregate_rows(hl.agg.counter(mt.vep.consequence_category))

2021-08-12 12:28:59 Hail: INFO: Coerced almost-sorted dataset
2021-08-12 12:29:00 Hail: INFO: Coerced almost-sorted dataset


frozendict({'synonymous': 29375, 'NA': 7731, 'damaging_missense': 5317, 'other_missense': 45901, 'non_coding': 66778, 'ptv': 1875})

In [97]:
mt.filter_rows(~hl.is_defined(mt.vep.consequence_category)).vep.show()

2021-08-12 12:24:12 Hail: INFO: Coerced almost-sorted dataset
2021-08-12 12:24:12 Hail: INFO: Coerced almost-sorted dataset


+---------------+------------+------------------+----------+---------------+------------+-------------------+----------+---------+------------------+------------+-------------------+----------+------------------+-------------+
| locus         | alleles    | vep.Feature_type | vep.SIFT | vep.CANONICAL | vep.INTRON | vep.PROVEAN_score | vep.EXON | vep.LoF | vep.PROVEAN_pred | vep.SYMBOL | vep.SYMBOL_SOURCE | vep.Gene | vep.CDS_position | vep.Feature |
+---------------+------------+------------------+----------+---------------+------------+-------------------+----------+---------+------------------+------------+-------------------+----------+------------------+-------------+
| locus<GRCh38> | array<str> | str              | str      | str           | str        | str               | str      | str     | str              | str        | str               | str      | str              | str         |
+---------------+------------+------------------+----------+---------------+------------+-------------------+----------+---------+------------------+------------+-------------------+----------+------------------+-------------+
+---------------+------------+------------------+----------+---------------+------------+-------------------+----------+---------+------------------+------------+-------------------+----------+------------------+-------------+

+-----------------+-------------------------+-------------------------+-------------+--------------------------+----------------------+--------------------------+-----------------+----------------------------+---------------------------+
| vep.Consequence | vep.Polyphen2_HDIV_pred | vep.MutationTaster_pred | vep.BIOTYPE | vep.Polyphen2_HDIV_score | vep.Protein_position | vep.Polyphen2_HVAR_score | vep.REVEL_score | vep.MutationAssessor_score | vep.MutationAssessor_pred |
+-----------------+-------------------------+-------------------------+-------------+--------------------------+----------------------+--------------------------+-----------------+----------------------------+---------------------------+
| str             | str                     | str                     | str         | str                      | str                  | str                      | str             | str                        | str                       |
+-----------------+-------------------------+-------------------------+-------------+--------------------------+----------------------+--------------------------+-----------------+----------------------------+---------------------------+
+-----------------+-------------------------+-------------------------+-------------+--------------------------+----------------------+--------------------------+-----------------+----------------------------+---------------------------+

+--------------+------------+-------------------------+----------------+---------------+--------------+---------------+----------------+---------------+--------------------------+----------------+---------------+-------------------------+
| vep.LRT_pred | vep.IMPACT | vep.Polyphen2_HVAR_pred | vep.SIFT_score | vep.LoF_flags | vep.CADD_raw | vep.SIFT_pred | vep.LoF_filter | vep.LRT_score | vep.MutationTaster_score | vep.CADD_phred | vep.sift_pred | vep.polyphen2_hdiv_pred |
+--------------+------------+-------------------------+----------------+---------------+--------------+---------------+----------------+---------------+--------------------------+----------------+---------------+-------------------------+
| str          | str        | str                     | str            | str           | str          | str           | str            | str           | str                      | str            | str           | str                     |
+--------------+------------+-------------------------+----------------+---------------+--------------+---------------+----------------+---------------+--------------------------+----------------+---------------+-------------------------+
+------------

In [107]:
mt.filter_rows(hl.literal('NA').contains(mt.vep.consequence_category)).vep.show()

2021-08-12 12:26:35 Hail: INFO: Coerced almost-sorted dataset
2021-08-12 12:26:35 Hail: INFO: Coerced almost-sorted dataset


+----------------+------------+------------------+----------------------------------+---------------+------------+-------------------+----------+---------+------------------+------------+-------------------+-------------------+
| locus          | alleles    | vep.Feature_type | vep.SIFT                         | vep.CANONICAL | vep.INTRON | vep.PROVEAN_score | vep.EXON | vep.LoF | vep.PROVEAN_pred | vep.SYMBOL | vep.SYMBOL_SOURCE | vep.Gene          |
+----------------+------------+------------------+----------------------------------+---------------+------------+-------------------+----------+---------+------------------+------------+-------------------+-------------------+
| locus<GRCh38>  | array<str> | str              | str                              | str           | str        | str               | str      | str     | str              | str        | str               | str               |
+----------------+------------+------------------+----------------------------------+---------------+------------+-------------------+----------+---------+------------------+------------+-------------------+-------------------+
| chr22:16784414 | ["T","G"]  | "Transcript"     | ""                               | "YES"         | "3/3"      | ""                | ""       | ""      | ""               | "XKR3"     | "HGNC"            | "ENSG00000172967" |
| chr22:16800028 | ["T","A"]  | "Transcript"     | ""                               | "YES"         | "2/3"      | ""                | ""       | ""      | ""               | "XKR3"     | "HGNC"            | "ENSG00000172967" |
| chr22:16962875 | ["G","A"]  | "Transcript"     | "deleterious(0)"                 | "YES"         | ""         | "-4.37&."         | "10/10"  | ""      | "D&."            | "GAB4"     | "HGNC"            | "ENSG00000215568" |
| chr22:16962880 | ["A","C"]  | "Transcript"     | ""                               | "YES"         | "9/9"      | ""                | ""       | ""      | ""               | "GAB4"     | "HGNC"            | "ENSG00000215568" |
| chr22:16962881 | ["G","A"]  | "Transcript"     | ""                               | "YES"         | "9/9"      | ""                | ""       | ""      | ""               | "GAB4"     | "HGNC"            | "ENSG00000215568" |
| chr22:16962882 | ["G","A"]  | "Transcript"     | ""                               | "YES"         | "9/9"      | ""                | ""       | ""      | ""               | "GAB4"     | "HGNC"            | "ENSG00000215568" |
| chr22:16962882 | ["G","T"]  | "Transcript"     | ""                               | "YES"         | "9/9"      | ""                | ""       | ""      | ""               | "GAB4"     | "HGNC"            | "ENSG00000215568" |
| chr22:16962883 | ["G","A"]  | "Transcript"     | ""                               | "YES"         | "9/9"      | ""                | ""       | ""      | ""               | "GAB4"     | "HGNC"            | "ENSG00000215568" |
| chr22:16962883 | ["G","T"]  | "Transcript"     | ""                               | "YES"         | "9/9"      | ""                | ""       | ""      | ""               | "GAB4"     | "HGNC"            | "ENSG00000215568" |
| chr22:16963718 | ["C","A"]  | "Transcript"     | ""                               | "YES"         | "9/9"      | ""                | ""       | ""      | ""               | "GAB4"     | "HGNC"            | "ENSG00000215568" |
| chr22:16963720 | ["G","A"]  | "Transcript"     | ""                               | "YES"         | "9/9"      | ""                | ""       | ""      | ""               | "GAB4"     | "HGNC"            | "ENSG00000215568" |
| chr22:16963827 | ["G","A"]  | "Transcript"     | ""                               | "YES"         | ""         | ""                | "9/10"   | ""      | ""               | "GAB4"     | "HGNC"            | "ENSG00000215568" |
| chr22:16964766 | ["C","T"]  | "Transcript"     | ""                               | "YES"         | ""         | ""       

In [30]:
#ptv_mt = mt.filter_rows(hl.literal('ptv').contains(mt.vep.consequence_category))
#burden_mt = construct_summary_mt(ptv_mt, gene_field = 'Gene')
ko_samples = extract_knockout_samples(mt)
ko_samples.show()


2021-08-13 14:51:18 Hail: INFO: Coerced almost-sorted dataset
2021-08-13 14:51:19 Hail: INFO: Coerced almost-sorted dataset
2021-08-13 14:51:22 Hail: INFO: Ordering unsorted dataset with network shuffle
2021-08-13 14:51:22 Hail: INFO: Coerced almost-sorted dataset
2021-08-13 14:51:23 Hail: INFO: Coerced almost-sorted dataset
2021-08-13 14:51:26 Hail: INFO: Ordering unsorted dataset with network shuffle


,,,
,,,genotypes
Gene,s,ko,rsid
str,str,str,str
"""ENSG00000008735""","""2702437""","""HO""","""chr22_50604850_G_A=1|1,chr22_50606762_C_T=1|1,chr22_50607005_C_T=1|1,chr22_50610353_T_G=1|1"""
"""ENSG00000054611""","""2702437""","""HO""","""chr22_46878742_C_T=1|1"""
"""ENSG00000054611""","""5710601""","""HO""","""chr22_46878742_C_T=1|1"""
"""ENSG00000056487""","""2702437""","""HO""","""chr22_44913773_C_T=1|1"""
"""ENSG00000056487""","""5710601""","""HO""","""chr22_44913773_C_T=1|1"""
"""ENSG00000069998""","""2702437""","""HO""","""chr22_17149596_C_A=1|1"""
"""ENSG00000069998""","""5710601""","""HO""","""chr22_17159155_G_A=1|1"""


2021-08-13 14:39:27 Hail: INFO: Coerced almost-sorted dataset
2021-08-13 14:39:27 Hail: INFO: Coerced almost-sorted dataset
2021-08-13 14:39:34 Hail: INFO: Ordering unsorted dataset with network shuffle
2021-08-13 14:39:35 Hail: INFO: Coerced almost-sorted dataset
2021-08-13 14:39:35 Hail: INFO: Coerced almost-sorted dataset
2021-08-13 14:39:42 Hail: INFO: Ordering unsorted dataset with network shuffle


,,,
,,,genotypes
Gene,s,ko,rsid
str,str,str,str
"""ENSG00000184113""","""5710601""","""HO""","""chr22_19524402_G_A=1|1"""
"""ENSG00000215568""","""2702437""","""HO""","""chr22_16988159_C_A=1|1"""


In [24]:
mt.rows().show()

2021-08-13 14:43:14 Hail: INFO: Coerced almost-sorted dataset
2021-08-13 14:43:14 Hail: INFO: Coerced almost-sorted dataset


+----------------+------------+----------------------+-----------+----------+----------------+---------+----------------+---------+------------+-------------------------+
| locus          | alleles    | rsid                 |      qual | filters  | info.AF        | info.AC | info.CM        | info.AN | vep.SYMBOL | vep.Polyphen2_HDIV_pred |
+----------------+------------+----------------------+-----------+----------+----------------+---------+----------------+---------+------------+-------------------------+
| locus<GRCh38>  | array<str> | str                  |   float64 | set<str> | array<float64> |   int32 | array<float64> |   int32 | str        | str                     |
+----------------+------------+----------------------+-----------+----------+----------------+---------+----------------+---------+------------+-------------------------+
| chr22:15528182 | ["C","T"]  | "chr22_15528182_C_T" | -1.00e+01 | NA       | [5.01e-06]     |       0 | [0.00e+00]     |       4 | "OR11H1"   | ""                      |
| chr22:15528188 | ["C","T"]  | "chr22_15528188_C_T" | -1.00e+01 | NA       | [1.75e-05]     |       0 | [7.07e-07]     |       4 | "OR11H1"   | ""                      |
| chr22:15528194 | ["G","A"]  | "chr22_15528194_G_A" | -1.00e+01 | NA       | [1.50e-05]     |       0 | [1.41e-06]     |       4 | "OR11H1"   | ".&."                   |
| chr22:15528206 | ["G","A"]  | "chr22_15528206_G_A" | -1.00e+01 | NA       | [1.25e-05]     |       0 | [2.83e-06]     |       4 | "OR11H1"   | ""                      |
| chr22:15528267 | ["A","G"]  | "chr22_15528267_A_G" | -1.00e+01 | NA       | [1.00e-05]     |       0 | [1.00e-05]     |       4 | "OR11H1"   | ".&."                   |
| chr22:15528300 | ["A","G"]  | "chr22_15528300_A_G" | -1.00e+01 | NA       | [1.00e-05]     |       0 | [1.39e-05]     |       4 | "OR11H1"   | ".&."                   |
| chr22:15528306 | ["T","A"]  | "chr22_15528306_T_A" | -1.00e+01 | NA       | [5.51e-05]     |       0 | [1.46e-05]     |       4 | "OR11H1"   | ".&."                   |
| chr22:15528309 | ["G","A"]  | "chr22_15528309_G_A" | -1.00e+01 | NA       | [5.01e-06]     |       0 | [1.50e-05]     |       4 | "OR11H1"   | ".&."                   |
| chr22:15528318 | ["A","G"]  | "chr22_15528318_A_G" | -1.00e+01 | NA       | [1.75e-05]     |       0 | [1.60e-05]     |       4 | "OR11H1"   | ".&."                   |
| chr22:15528340 | ["C","T"]  | "chr22_15528340_C_T" | -1.00e+01 | NA       | [5.01e-06]     |       0 | [1.86e-05]     |       4 | "OR11H1"   | ".&."                   |
| chr22:15528345 | ["G","T"]  | "chr22_15528345_G_T" | -1.00e+01 | NA       | [1.40e-04]     |       0 | [1.92e-05]     |       4 | "OR11H1"   | ".&."                   |
| chr22:15528351 | ["T","C"]  | "chr22_15528351_T_C" | -1.00e+01 | NA       | [5.76e-05]     |       0 | [1.99e-05]     |       4 | "OR11H1"   | ".&."                   |
| chr22:15528363 | ["C","T"]  | "chr22_15528363_C_T" | -1.00e+01 | NA       | [5.01e-06]     |       0 | [2.13e-05]     |       4 | "OR11H1"   | ".&."                   |
| chr22:15528413 | ["G","A"]  | "chr22_15528413_G_A" | -1.00e+01 | NA       | [1.25e-05]     |       0 | [2.72e-05]     |       4 | "OR11H1"   | ""                      |
| chr22:15528586 | ["G","A"]  | "chr22_15528586_G_A" | -1.00e+01 | NA       | [1.00e-05]     |       0 | [4.76e-05]     |       4 | "OR11H1"   | ".&."                   |
| chr22:15528586 | ["G","C"]  | "chr22_15528586_G_C" | -1.00e+01 | NA       | [5.01e-06]     |       0 | [4.76e-05]     |       4 | "OR11H1"   | ".&."                   |
| chr22:15528591 | ["T","C"]  | "chr22_15528591_T_C" | -1.00e+01 | NA       | [5.01e-06]     |       0 | [4.82e-05]     |       4 | "OR11H1"   | ""                      |
| chr22:15528594 | ["C","T"]  | "chr22_15528594_C_T" | -1.00e+01 | NA       | [5.01e-06]     |       0 | [4.86e-05]     |       4 | "OR11H1"   | ".&."                   |
| chr22:15528617 | ["G","A"]  | "chr22_15528617_G_A" | -1.00e+01 | 

In [155]:
# Open file containing VEP fields
with open('data/vep/vep_fields.txt', 'r') as file:
    fields = file.read().strip().split(',')
vep_path = 'data/vep/output/ukb_wes_200k_vep_chr22.vcf'
ht = hl.import_vcf(vep_path).rename({'info':'vep'}) 

# Add VEP fields by iteration
for i in range(len(fields)):
    ht = ht.annotate_rows(
        vep=ht.vep.annotate(
            col=ht.vep.CSQ.map(lambda x: (x.split('\\|')[i]))[0]
            ).rename({'col':f'{fields[i]}'})
    )

# Worst consequence is always first:
ht = ht.annotate_rows(vep = ht.vep.annotate(most_severe_consequence = ht.vep.Consequence.split('&')[0]))

# Extract various categories annotations and change type
ht = ht.annotate_rows(vep = ht.vep.annotate(sift_pred = ht.vep.SIFT_pred.split('&')[0]))
ht = ht.annotate_rows(vep = ht.vep.annotate(polyphen2_hdiv_pred = ht.vep.Polyphen2_HDIV_pred.split('&')[0]))
ht = ht.annotate_rows(vep = ht.vep.annotate(polyphen2_hvar_pred = ht.vep.Polyphen2_HVAR_pred.split('&')[0]))
ht = ht.annotate_rows(vep = ht.vep.annotate(cadd_phred_score = hl.parse_float(ht.vep.CADD_phred)))
ht = ht.annotate_rows(vep = ht.vep.annotate(revel_score = hl.parse_float(ht.vep.REVEL_score)))
    
# Define protein truncating variants
ptv = hl.set(["transcript_ablation", "splice_acceptor_variant",
              "splice_donor_variant", "stop_gained", "frameshift_variant"])
    
# Define missense variation
missense = hl.set(["stop_lost", "start_lost", "transcript_amplification",
                   "inframe_insertion", "inframe_deletion", "missense_variant",
                   "protein_altering_variant", "splice_region_variant"])
    
# Define synonymous
synonymous = hl.set(["incomplete_terminal_codon_variant", "stop_retained_variant", "synonymous_variant"])
    
# Define non coding variation
non_coding = hl.set(["coding_sequence_variant", "mature_miRNA_variant", "5_prime_UTR_variant",
              "3_prime_UTR_variant", "non_coding_transcript_exon_variant", "intron_variant",
              "NMD_transcript_variant", "non_coding_transcript_variant", "upstream_gene_variant",
              "downstream_gene_variant", "TFBS_ablation", "TFBS_amplification", "TF_binding_site_variant",
              "regulatory_region_ablation", "regulatory_region_amplification", "feature_elongation",
              "regulatory_region_variant", "feature_truncation", "intergenic_variant"])
    
    # Create categories for downstream analysis
ht = ht.annotate_rows(vep = ht.vep.annotate(consequence_category = 
    hl.case().when(ptv.contains(ht.vep.Consequence), "ptv")
        .when(missense.contains(ht.vep.Consequence) & 
                (~hl.is_defined(ht.vep.cadd_phred_score) | 
                ~hl.is_defined(ht.vep.revel_score)), "other_missense")                                   
        .when(missense.contains(ht.vep.Consequence) & 
                (ht.vep.cadd_phred_score >= 20) & 
                (ht.vep.revel_score >= 0.6), "damaging_missense") 
        .when(missense.contains(ht.vep.Consequence), "other_missense")
        .when(synonymous.contains(ht.vep.Consequence), "synonymous")
        .when(non_coding.contains(ht.vep.Consequence), "non_coding")
        .default("NA")
    ))

In [156]:
ht2 = ht.filter_rows(ht.vep.consequence_category == 'NA')

In [165]:
ht2.aggregate_rows(hl.agg.counter(ht2.vep.most_severe_consequence))
ht2.aggregate_rows(hl.agg.counter(ht2.vep.Consequence))

2021-08-15 17:24:31 Hail: INFO: Coerced almost-sorted dataset
2021-08-15 17:24:37 Hail: INFO: Coerced almost-sorted dataset


frozendict({'splice_region_variant&intron_variant&non_coding_transcript_variant': 12, 'splice_region_variant&synonymous_variant': 732, 'missense_variant&splice_region_variant': 1341, 'splice_region_variant&stop_retained_variant': 1, 'splice_region_variant&3_prime_UTR_variant': 6, 'start_lost&splice_region_variant': 2, 'splice_acceptor_variant&non_coding_transcript_variant': 1, 'splice_region_variant&3_prime_UTR_variant&NMD_transcript_variant': 4, 'intron_variant&non_coding_transcript_variant': 113, 'splice_region_variant&non_coding_transcript_exon_variant': 5, 'stop_gained&splice_region_variant': 51, 'splice_region_variant&intron_variant&NMD_transcript_variant': 8, 'splice_donor_variant&NMD_transcript_variant': 2, 'splice_acceptor_variant&NMD_transcript_variant': 1, 'splice_region_variant&intron_variant': 4629, '3_prime_UTR_variant&NMD_transcript_variant': 595, 'intron_variant&NMD_transcript_variant': 188, 'splice_region_variant&5_prime_UTR_variant': 40})

In [ ]:
(hl.case()
...   .when(x < -1, 1)
...   .when((x >= -1) & (x <= 2), 2)
...   .when(x > 2, 3)
...   .or_missing())
<Int32Expression of type int32>

frozenset({'inframe_deletion',
           'inframe_insertion',
           'missense_variant',
           'protein_altering_variant',
           'splice_region_variant',
           'start_lost',
           'stop_lost',
           'transcript_amplification'})